<a href="https://githubtocolab.com/gee-community/geemap/blob/master/docs/notebooks/123_sentinel1_timelapse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

**Creating Sentinel-1 SAR imagery timelapse**

Uncomment the following line to install [geemap](https://geemap.org) if needed.

In [ ]:
# !pip install -U geemap

In [1]:
import ee
import geemap

In [5]:
ee.Authenticate()  # 认证登录 GEE
ee.Initialize(project='ee-luyuxiang4917')  # 初始化项目，替换成 GEE 项目 ID

In [6]:
Map = geemap.Map()
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

Pan and zoom to an area of interest and draw a rectangle on the map.

In [7]:
roi = Map.user_roi
if roi is None:
    roi = ee.Geometry.BBox(117.1132, 3.5227, 117.2214, 3.5843)
    Map.addLayer(roi)
    Map.centerObject(roi)

In [13]:
timelapse = geemap.sentinel1_timelapse(
    roi,
    out_gif="kaohsiung_port_s1_yearly.gif",
    start_year=2015,
    end_year=2025,
    start_date="01-01",
    end_date="12-31",
    frequency="year",
    bands=["VV"],
    vis_params={"min": -25, "max": 0},
    palette="Greys",
    frames_per_second=1,
    title="Kaohsiung Port Sentinel-1 Annual Timelapse",
    add_colorbar=True,
    colorbar_bg_color="gray",
)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/kaohsiung_port_s1_yearly.gif


In [14]:
geemap.show_image(timelapse)

Output()

In [37]:
timelapse_s2 = geemap.sentinel2_timelapse(
    roi,
    out_gif="kaohsiung_port_s2_clean.gif",
    start_year=2015,
    end_year=2020,
    start_date="01-01",
    end_date="12-31",
    frequency="year",
    bands=["Red", "Green", "Blue"],
    vis_params={
    "min": 0.04,
    "max": 0.20,
    "gamma": 1.15
    },
    frames_per_second=1,
    title="Kaohsiung Port Sentinel-2 Clean Annual Timelapse"
)

Generating URL...
User memory limit exceeded.
User memory limit exceeded.


In [36]:
geemap.show_image(timelapse_s2)

Output()

In [44]:
from pathlib import Path
from PIL import Image, ImageSequence

In [46]:
# =============================
# 2. 输出目录
# =============================
out_dir = Path("./s2_timelapse_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# =============================
# 3. 分段年份
# S2_SR_HARMONIZED 从 2017-03-28 开始
# =============================
year_ranges = [
    (2017, 2018),
    (2019, 2020),
    (2021, 2022),
    (2023, 2024),
    (2025, 2025),
]

generated_files = []

# =============================
# 4. 基础 Sentinel-2 集合
# 只按云量筛选
# =============================
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(roi)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

# =============================
# 5. 安全生成单年帧
# 如果某一年没数据，就跳过
# =============================
def build_year_frame(year):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)

    yearly = s2.filterDate(start, end)

    # 判断这一年是否有影像
    count = yearly.size().getInfo()
    if count == 0:
        print(f"  {year}: no images, skipped")
        return None

    img = (
        yearly.median()
        .clip(roi)
        .divide(10000)
        .visualize(
            bands=["B4", "B3", "B2"],
            min=0.04,
            max=0.20,
            gamma=1.15,
        )
        .set("system:time_start", start.millis())
    )
    print(f"  {year}: {count} images")
    return img

# =============================
# 6. 分段生成 GIF
# 只有真正成功时才加入 generated_files
# =============================
for start_year, end_year in year_ranges:
    out_gif = out_dir / f"kaohsiung_port_s2_{start_year}_{end_year}.gif"
    print(f"\nGenerating {start_year}-{end_year} ...")

    frames = []
    for year in range(start_year, end_year + 1):
        frame = build_year_frame(year)
        if frame is not None:
            frames.append(frame)

    if len(frames) == 0:
        print(f"  No valid frames for {start_year}-{end_year}, skipped")
        continue

    col = ee.ImageCollection(frames)

    try:
        geemap.download_ee_video(
            col,
            {
                "region": roi,
                "dimensions": 600,      # 比 700 更稳
                "framesPerSecond": 1,
                "crs": "EPSG:3857",
            },
            str(out_gif),
        )

        if out_gif.exists():
            generated_files.append(str(out_gif))
            print(f"Saved: {out_gif}")
        else:
            print(f"Download finished but file not found: {out_gif}")

    except Exception as e:
        print(f"Failed for {start_year}-{end_year}: {e}")

print("\nSuccessfully generated GIFs:")
for f in generated_files:
    print(f)

# =============================
# 7. 合并多个 GIF
# 只合并真实存在的文件
# =============================
merged_output = out_dir / "kaohsiung_port_s2_2017_2025_merged.gif"

all_frames = []
base_size = None

for gif_path in generated_files:
    p = Path(gif_path)
    if not p.exists():
        print(f"Missing file, skipped: {gif_path}")
        continue

    img = Image.open(p)
    for frame in ImageSequence.Iterator(img):
        frame_rgba = frame.convert("RGBA")

        if base_size is None:
            base_size = frame_rgba.size

        if frame_rgba.size != base_size:
            frame_rgba = frame_rgba.resize(base_size)

        all_frames.append(frame_rgba)

if not all_frames:
    raise RuntimeError("No valid GIF frames found to merge.")

all_frames[0].save(
    merged_output,
    save_all=True,
    append_images=all_frames[1:],
    duration=1000,   # 1秒1帧
    loop=0,
    disposal=2
)

print(f"\nMerged GIF saved to: {merged_output}")


Generating 2017-2018 ...
  2017: no images, skipped
  2018: 5 images
Generating URL...
Please wait ...
The GIF image has been saved to: /content/s2_timelapse_outputs/kaohsiung_port_s2_2017_2018.gif
Saved: s2_timelapse_outputs/kaohsiung_port_s2_2017_2018.gif

Generating 2019-2020 ...
  2019: 91 images
  2020: 96 images
Generating URL...
Please wait ...
The GIF image has been saved to: /content/s2_timelapse_outputs/kaohsiung_port_s2_2019_2020.gif
Saved: s2_timelapse_outputs/kaohsiung_port_s2_2019_2020.gif

Generating 2021-2022 ...
  2021: 108 images
  2022: 65 images
Generating URL...
Please wait ...
The GIF image has been saved to: /content/s2_timelapse_outputs/kaohsiung_port_s2_2021_2022.gif
Saved: s2_timelapse_outputs/kaohsiung_port_s2_2021_2022.gif

Generating 2023-2024 ...
  2023: 63 images
  2024: 77 images
Generating URL...
Please wait ...
The GIF image has been saved to: /content/s2_timelapse_outputs/kaohsiung_port_s2_2023_2024.gif
Saved: s2_timelapse_outputs/kaohsiung_port_s2_2

In [49]:
from PIL import Image, ImageSequence, ImageDraw, ImageFont

input_gif = "./s2_timelapse_outputs/kaohsiung_port_s2_2017_2025_merged.gif"
output_gif = "./s2_timelapse_outputs/kaohsiung_port_s2_2017_2025_merged_labeled.gif"

start_year = 2018

img = Image.open(input_gif)
frames = [frame.convert("RGBA") for frame in ImageSequence.Iterator(img)]

print("frame count =", len(frames))

try:
    font = ImageFont.truetype("arial.ttf", 32)
except:
    font = ImageFont.load_default()

labeled_frames = []

for i, frame in enumerate(frames):
    year = start_year + i
    draw = ImageDraw.Draw(frame)

    text = str(year)
    x, y = 20, 20

    bbox = draw.textbbox((x, y), text, font=font)
    pad = 10

    draw.rounded_rectangle(
        (bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad),
        radius=8,
        fill=(0, 0, 0, 160)
    )
    draw.text((x, y), text, font=font, fill=(255, 255, 255, 255))

    labeled_frames.append(frame)

duration = img.info.get("duration", 1000)

labeled_frames[0].save(
    output_gif,
    save_all=True,
    append_images=labeled_frames[1:],
    duration=duration,
    loop=0,
    disposal=2
)

print(f"Saved labeled GIF to: {output_gif}")

frame count = 8
Saved labeled GIF to: ./s2_timelapse_outputs/kaohsiung_port_s2_2017_2025_merged_labeled.gif


In [48]:
from PIL import Image, ImageSequence
from pathlib import Path

gif_files = [
    "./s2_timelapse_outputs/kaohsiung_port_s2_2017_2018.gif",
    "./s2_timelapse_outputs/kaohsiung_port_s2_2019_2020.gif",
    "./s2_timelapse_outputs/kaohsiung_port_s2_2021_2022.gif",
    "./s2_timelapse_outputs/kaohsiung_port_s2_2023_2024.gif",
    "./s2_timelapse_outputs/kaohsiung_port_s2_2025_2025.gif",
    "./s2_timelapse_outputs/kaohsiung_port_s2_2017_2025_merged.gif",
]

for gif_path in gif_files:
    p = Path(gif_path)
    if not p.exists():
        print(f"{p.name}: missing")
        continue

    img = Image.open(p)
    n = sum(1 for _ in ImageSequence.Iterator(img))
    print(f"{p.name}: {n} frame(s)")

kaohsiung_port_s2_2017_2018.gif: 1 frame(s)
kaohsiung_port_s2_2019_2020.gif: 2 frame(s)
kaohsiung_port_s2_2021_2022.gif: 2 frame(s)
kaohsiung_port_s2_2023_2024.gif: 2 frame(s)
kaohsiung_port_s2_2025_2025.gif: 1 frame(s)
kaohsiung_port_s2_2017_2025_merged.gif: 8 frame(s)
